# Model Comparison — F1 Metric

Leave-one-athlete-out CV results for all models on the **train set**, final model on the **test set**.

Two evaluation modes:
1. **No postprocessing** — raw argmax of model probabilities
2. **Viterbi** — HMM Viterbi decoding using transition matrix estimated from training labels; kinematic detector uses min-duration filter (3 frames) since it outputs labels, not probabilities

Each bar chart and each confusion matrix is saved as a separate PNG.

In [1]:
import joblib
import json
import os

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import xgboost as xgb

from experiments.gait_detection.config import ExperimentConfig
from src.gait.detection.detectors import KinematicDetector
from src.gait.detection.metrics import per_class_f1, confusion_matrix
from src.gait.detection.postprocess import (
    estimate_hmm_params,
    min_duration_filter,
    viterbi_decode,
)
from src.gait.gait_data.dataset import load_dataset, train_test_split
from src.gait.image.dataset import load_image_dataset

XGB_PARAMS_PATH       = "data/output/gait/pose_data/stage3/best_params.json"
IMG_XGB_MODEL_PATH    = "data/output/gait/image/stage1/checkpoints/xgboost_best.pkl"

XGB_LOAO_MANIFEST     = "data/output/gait/pose_data/xgb_loao/manifest.json"
IMG_XGB_LOAO_MANIFEST = "data/output/gait/image/xgb_loao/manifest.json"
TCN_LOAO_MANIFEST     = "data/output/gait/pose_data/stage4/manifest.json"
IMG_TCN_LOAO_MANIFEST = "data/output/gait/image/stage4/manifest.json"
RESNET_LOAO_MANIFEST  = "data/output/gait/image/resnet/loao/manifest.json"

TCN_INFER_MANIFEST     = "data/output/gait/pose_data/stage6/manifest.json"
IMG_TCN_INFER_MANIFEST = "data/output/gait/image/stage6/manifest.json"
RESNET_INFER_MANIFEST  = "data/output/gait/image/resnet/infer/manifest.json"

PLOTS_DIR = "data/output/gait/plots/compare_kinematic_xgboost"
os.chdir('..')
os.makedirs(PLOTS_DIR, exist_ok=True)

matplotlib.use('Agg')


In [2]:
cfg = ExperimentConfig()
all_records = load_dataset(cfg.annotations_csv, fps=cfg.fps)
train_records, test_records, TEST_ATHLETES = train_test_split(all_records)

print(f"{len(all_records)} total, {len(train_records)} train, {len(test_records)} test")
print(f"Test athletes: {TEST_ATHLETES}")

IMG_FEATURES_DIR = "data/output/gait/image_features"
VIDEO_INPUT_DIR  = "data/input/optojump"
img_all = load_image_dataset(cfg.annotations_csv, IMG_FEATURES_DIR, VIDEO_INPUT_DIR)
img_by_path = {r.video_path: r for r in img_all}
print(f"{len(img_all)} image records loaded")

178 total, 162 train, 16 test
Test athletes: ['antoni_pejtler', 'milosz_jarzab']
178 image records loaded


In [3]:
with open(XGB_PARAMS_PATH) as f:
    xgb_meta = json.load(f)
XGB_PARAMS  = xgb_meta["best_params"]
FEATURE_IDX = xgb_meta.get("feature_idx")
print(f"XGBoost params loaded (feature_idx={FEATURE_IDX})")

CLASSES = cfg.class_names + ["macro"]

MODEL_META = [
    {"key": "kinematic", "label": "Kinematic",   "color": "C0"},
    {"key": "xgboost",   "label": "XGBoost",     "color": "C1"},
    {"key": "img_xgb",   "label": "Img XGBoost", "color": "C5"},
    {"key": "tcn",       "label": "Pose TCN",    "color": "C2"},
    {"key": "img_tcn",   "label": "Img TCN",     "color": "C3"},
    {"key": "resnet",    "label": "ResNet",       "color": "C4"},
]


def flatten(records, feature_idx=None):
    X, y = [], []
    for r in records:
        feats = r.features if feature_idx is None else r.features[:, feature_idx]
        X.append(feats)
        y.append(r.labels)
    return np.vstack(X), np.concatenate(y)


def eval_result(pred_list, rec_list):
    y_true = np.concatenate([r.labels for r in rec_list])
    y_pred = np.concatenate(pred_list)
    return {
        "f1": per_class_f1(y_true, y_pred, cfg.n_classes, cfg.class_names),
        "cm": confusion_matrix(y_true, y_pred, cfg.n_classes),
    }


def per_athlete_f1(pred_list, rec_list):
    """Return {class: [f1_per_athlete]} by grouping records by athlete."""
    from collections import defaultdict
    groups = defaultdict(lambda: {"preds": [], "labels": []})
    for pred, rec in zip(pred_list, rec_list):
        groups[rec.athlete]["preds"].append(pred)
        groups[rec.athlete]["labels"].append(rec.labels)
    result = {c: [] for c in CLASSES}
    for g in groups.values():
        y_true = np.concatenate(g["labels"])
        y_pred = np.concatenate(g["preds"])
        f1 = per_class_f1(y_true, y_pred, cfg.n_classes, cfg.class_names)
        for c in CLASSES:
            result[c].append(f1[c])
    return result


def _load_manifest_probs(manifest_path, ref_records, split=None, img_lookup=None):
    """Load probs from a manifest. split=None accepts all records.
    Pass img_lookup=img_by_path for image-based models so that record labels
    align with the image frame count (which may differ from pose frame count)."""
    if not os.path.exists(manifest_path):
        print(f"  missing: {manifest_path}")
        return None, None
    with open(manifest_path) as f:
        raw = json.load(f)
    idx = {
        r["video_path"]: r for r in raw["records"]
        if split is None or r["split"] == split
    }
    recs, probs = [], []
    for r in ref_records:
        entry = idx.get(r.video_path)
        if entry is None:
            continue
        target = img_lookup.get(r.video_path) if img_lookup else r
        if target is None:
            continue
        recs.append(target)
        probs.append(np.load(entry["probs_path"]))
    return (recs, probs) if recs else (None, None)

XGBoost params loaded (feature_idx=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19])


In [4]:
train_data = {}

# Kinematic — deterministic, produces labels not probs
kinematic = KinematicDetector()
train_data["kinematic"] = {
    "records": train_records,
    "labels":  [kinematic.predict(r.features, cfg.fps) for r in train_records],
}

# XGBoost pose — LOAO probs (pose records, labels match pose frame count)
recs, probs = _load_manifest_probs(XGB_LOAO_MANIFEST, train_records)
if recs:
    train_data["xgboost"] = {"records": recs, "probs": probs}
    print(f"XGBoost LOAO: {len(recs)} records")

# Image XGBoost — LOAO probs (use image records so labels match image frame count)
recs, probs = _load_manifest_probs(IMG_XGB_LOAO_MANIFEST, train_records, img_lookup=img_by_path)
if recs:
    train_data["img_xgb"] = {"records": recs, "probs": probs}
    print(f"Img XGBoost LOAO: {len(recs)} records")

# Pose TCN — LOAO probs
recs, probs = _load_manifest_probs(TCN_LOAO_MANIFEST, train_records)
if recs:
    train_data["tcn"] = {"records": recs, "probs": probs}
    print(f"Pose TCN LOAO: {len(recs)} records")

# Image TCN — LOAO probs (use image records)
recs, probs = _load_manifest_probs(IMG_TCN_LOAO_MANIFEST, train_records, img_lookup=img_by_path)
if recs:
    train_data["img_tcn"] = {"records": recs, "probs": probs}
    print(f"Img TCN LOAO: {len(recs)} records")

# ResNet — LOAO probs (use image records)
recs, probs = _load_manifest_probs(RESNET_LOAO_MANIFEST, train_records, img_lookup=img_by_path)
if recs:
    train_data["resnet"] = {"records": recs, "probs": probs}
    print(f"ResNet LOAO: {len(recs)} records")

XGBoost LOAO: 162 records
Img XGBoost LOAO: 162 records
Pose TCN LOAO: 162 records
Img TCN LOAO: 162 records
ResNet LOAO: 162 records


In [5]:
test_data = {}

# Kinematic
test_data["kinematic"] = {
    "records": test_records,
    "labels":  [kinematic.predict(r.features, cfg.fps) for r in test_records],
}

# XGBoost pose — train final model on all train records, predict probs on test
X_train, y_train = flatten(train_records, FEATURE_IDX)
clf_final = xgb.XGBClassifier(
    **XGB_PARAMS,
    eval_metric="mlogloss", tree_method="hist", device="cpu", verbosity=0,
)
clf_final.fit(X_train, y_train)
test_data["xgboost"] = {
    "records": test_records,
    "probs": [
        clf_final.predict_proba(
            (r.features if FEATURE_IDX is None else r.features[:, FEATURE_IDX]).astype(np.float32)
        ).astype(np.float32)
        for r in test_records
    ],
}
print("XGBoost trained on all train records")

# Image XGBoost — load pkl model, predict probs on test image features
if os.path.exists(IMG_XGB_MODEL_PATH):
    img_xgb_clf = joblib.load(IMG_XGB_MODEL_PATH)
    img_recs, img_probs = [], []
    for r in test_records:
        img_r = img_by_path.get(r.video_path)
        if img_r is None:
            continue
        img_recs.append(img_r)  # image record so labels match image frame count
        img_probs.append(
            img_xgb_clf.predict_proba(img_r.features.astype(np.float32)).astype(np.float32)
        )
    if img_recs:
        test_data["img_xgb"] = {"records": img_recs, "probs": img_probs}
        print(f"Img XGBoost: {len(img_recs)} test records")
else:
    print(f"Img XGBoost model not found: {IMG_XGB_MODEL_PATH}")

# Pose TCN — inference manifest (split=test, pose records)
recs, probs = _load_manifest_probs(TCN_INFER_MANIFEST, test_records, split="test")
if recs:
    test_data["tcn"] = {"records": recs, "probs": probs}
    print(f"Pose TCN: {len(recs)} test records")

# Image TCN — inference manifest (split=test, use image records)
recs, probs = _load_manifest_probs(IMG_TCN_INFER_MANIFEST, test_records, split="test", img_lookup=img_by_path)
if recs:
    test_data["img_tcn"] = {"records": recs, "probs": probs}
    print(f"Img TCN: {len(recs)} test records")

# ResNet — inference manifest (split=test, use image records)
recs, probs = _load_manifest_probs(RESNET_INFER_MANIFEST, test_records, split="test", img_lookup=img_by_path)
if recs:
    test_data["resnet"] = {"records": recs, "probs": probs}
    print(f"ResNet: {len(recs)} test records")

XGBoost trained on all train records
Img XGBoost: 16 test records
Pose TCN: 16 test records
Img TCN: 16 test records
ResNet: 16 test records


In [6]:
all_train_labels = [r.labels for r in train_records]
startprob, transmat = estimate_hmm_params(all_train_labels)

STATE_NAMES = ["S_L", "F_L→R", "S_R", "F_R→L"]
import pandas as pd
print("HMM startprob:")
print(pd.Series(np.round(startprob, 3), index=STATE_NAMES).to_string())
print("\nHMM transmat:")
print(pd.DataFrame(np.round(transmat, 3), index=STATE_NAMES, columns=STATE_NAMES).to_string())

HMM startprob:
S_L      0.019
F_L→R    0.481
S_R      0.019
F_R→L    0.481

HMM transmat:
         S_L  F_L→R    S_R  F_R→L
S_L    0.941  0.059  0.000  0.000
F_L→R  0.021  0.920  0.059  0.000
S_R    0.000  0.000  0.941  0.059
F_R→L  0.059  0.000  0.021  0.920


In [7]:
def build_methods(data, use_viterbi=False):
    methods = []
    for meta in MODEL_META:
        key = meta["key"]
        if key not in data:
            continue
        entry = data[key]

        if key == "kinematic":
            labels = entry["labels"]
            preds = (
                [min_duration_filter(l, min_frames=3) for l in labels]
                if use_viterbi else labels
            )
        else:
            probs_list = entry["probs"]
            preds = (
                [viterbi_decode(p, transmat, startprob) for p in probs_list]
                if use_viterbi
                else [p.argmax(axis=1).astype(np.int64) for p in probs_list]
            )

        res = eval_result(preds, entry["records"])
        af1 = per_athlete_f1(preds, entry["records"])
        methods.append({
            "key":        key,
            "label":      meta["label"],
            "color":      meta["color"],
            "f1":         res["f1"],
            "cm":         res["cm"],
            "athlete_f1": af1,
        })
    return methods


def _draw_bars(ax, methods, show_strips):
    rng = np.random.default_rng(42)
    n   = len(methods)
    w   = 0.7 / n
    x   = np.arange(len(CLASSES))

    for i, m in enumerate(methods):
        offset = (i - n / 2 + 0.5) * w
        af1    = m["athlete_f1"]
        means  = [np.mean(af1[c]) for c in CLASSES]
        stds   = [np.std(af1[c])  for c in CLASSES]

        ax.bar(x + offset, means, w, yerr=stds, capsize=3,
               label=m["label"], color=m["color"], alpha=0.65,
               error_kw={"elinewidth": 1.2, "capthick": 1.2})

        for j, (c, mean, std) in enumerate(zip(CLASSES, means, stds)):
            if show_strips:
                vals   = np.array(af1[c])
                jitter = rng.uniform(-w * 0.22, w * 0.22, size=len(vals))
                ax.scatter(x[j] + offset + jitter, vals,
                           color=m["color"], s=14, alpha=0.9, zorder=3, linewidths=0)
            ax.text(x[j] + offset, mean + std + 0.025,
                    f"{mean:.2f}", ha="center", va="bottom", fontsize=8)

    ax.set_xticks(x)
    ax.set_xticklabels([c.replace("_", "\n") for c in CLASSES], fontsize=10)
    ax.set_ylabel("F1 score")
    ax.set_ylim(0, 1.2)
    ax.legend(loc="lower right", fontsize=8)
    ax.grid(axis="y", alpha=0.3)


def plot_f1_bars(methods, title, save_path):
    fig, ax = plt.subplots(figsize=(12, 5))
    _draw_bars(ax, methods, show_strips=False)
    #ax.set_title(title)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()


def plot_f1_strip(methods, title, save_path):
    fig, ax = plt.subplots(figsize=(12, 5))
    _draw_bars(ax, methods, show_strips=True)
    #ax.set_title(title)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()


def f1_table(methods):
    rows = []
    for c in CLASSES:
        row = {"Class": c.replace("_", " ")}
        for m in methods:
            af1  = m["athlete_f1"][c]
            mean = np.mean(af1)
            std  = np.std(af1)
            row[m["label"]] = f"{mean:.3f} ± {std:.3f}"
        rows.append(row)
    return pd.DataFrame(rows).set_index("Class")


def plot_confusion_matrix(method, title, save_path):
    labels_short = ["L-stance", "R-stance", "Flight"]
    cm_arr  = np.array(method["cm"], dtype=float)
    cm_norm = cm_arr / cm_arr.sum(axis=1, keepdims=True)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm_norm, annot=True, fmt=".2f", ax=ax,
                xticklabels=labels_short, yticklabels=labels_short,
                cmap="Blues", vmin=0, vmax=1, cbar=False)
    ax.set_title(f"{method['label']} — {title}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()


def plot_all(methods, section, split, title_prefix):
    plot_f1_bars(
        methods,
        f"{title_prefix} — F1",
        os.path.join(PLOTS_DIR, f"{section}_{split}_f1_bars.png"),
    )
    plot_f1_strip(
        methods,
        f"{title_prefix} — F1 (per athlete)",
        os.path.join(PLOTS_DIR, f"{section}_{split}_f1_strip.png"),
    )
    display(f1_table(methods))
    for m in methods:
        plot_confusion_matrix(
            m,
            title_prefix,
            os.path.join(PLOTS_DIR, f"{section}_{split}_cm_{m['key']}.png"),
        )

---
## No Postprocessing

Raw argmax of model probabilities. Kinematic detector output used directly.

In [10]:
methods_raw_train = build_methods(train_data, use_viterbi=False)
methods_raw_test  = build_methods(test_data,  use_viterbi=False)

plot_all(methods_raw_train, "no_pp", "train", "Train (LOAO) \u2014 No Postprocessing")
plot_all(methods_raw_test,  "no_pp", "test",  f"Test ({', '.join(TEST_ATHLETES)}) \u2014 No Postprocessing")

,Kinematic,XGBoost,Img XGBoost,Pose TCN,Img TCN,ResNet
Class,,,,,,
left stance,0.668 ± 0.083,0.842 ± 0.079,0.641 ± 0.107,0.849 ± 0.081,0.830 ± 0.082,0.823 ± 0.091
right stance,0.702 ± 0.095,0.851 ± 0.082,0.696 ± 0.106,0.856 ± 0.077,0.820 ± 0.098,0.816 ± 0.093
flight,0.592 ± 0.070,0.863 ± 0.031,0.817 ± 0.061,0.853 ± 0.042,0.846 ± 0.053,0.870 ± 0.054
macro,0.654 ± 0.071,0.852 ± 0.059,0.718 ± 0.076,0.853 ± 0.062,0.832 ± 0.069,0.836 ± 0.070


,Kinematic,XGBoost,Img XGBoost,Pose TCN,Img TCN,ResNet
Class,,,,,,
left stance,0.714 ± 0.008,0.910 ± 0.020,0.652 ± 0.075,0.924 ± 0.011,0.764 ± 0.036,0.862 ± 0.017
right stance,0.751 ± 0.036,0.938 ± 0.018,0.662 ± 0.061,0.925 ± 0.005,0.807 ± 0.025,0.809 ± 0.055
flight,0.614 ± 0.019,0.909 ± 0.028,0.822 ± 0.033,0.908 ± 0.016,0.920 ± 0.023,0.880 ± 0.017
macro,0.693 ± 0.016,0.919 ± 0.022,0.712 ± 0.016,0.919 ± 0.011,0.830 ± 0.004,0.850 ± 0.007


---
## Viterbi Postprocessing

HMM Viterbi decoding using transition matrix estimated from training labels.
Kinematic detector uses min-duration filter (3 frames) as it produces labels, not probabilities.

In [9]:
methods_viterbi_train = build_methods(train_data, use_viterbi=True)
methods_viterbi_test  = build_methods(test_data,  use_viterbi=True)

plot_all(methods_viterbi_train, "viterbi", "train", "Train (LOAO) \u2014 Viterbi")
plot_all(methods_viterbi_test,  "viterbi", "test",  f"Test ({', '.join(TEST_ATHLETES)}) \u2014 Viterbi")

,Kinematic,XGBoost,Img XGBoost,Pose TCN,Img TCN,ResNet
Class,,,,,,
left stance,0.668 ± 0.083,0.857 ± 0.073,0.715 ± 0.134,0.855 ± 0.075,0.834 ± 0.087,0.842 ± 0.088
right stance,0.702 ± 0.095,0.862 ± 0.079,0.755 ± 0.114,0.860 ± 0.075,0.827 ± 0.104,0.832 ± 0.092
flight,0.592 ± 0.070,0.867 ± 0.026,0.825 ± 0.062,0.855 ± 0.038,0.847 ± 0.053,0.872 ± 0.055
macro,0.654 ± 0.071,0.862 ± 0.054,0.765 ± 0.088,0.857 ± 0.056,0.836 ± 0.073,0.849 ± 0.068


,Kinematic,XGBoost,Img XGBoost,Pose TCN,Img TCN,ResNet
Class,,,,,,
left stance,0.714 ± 0.008,0.912 ± 0.018,0.745 ± 0.131,0.921 ± 0.012,0.765 ± 0.048,0.872 ± 0.020
right stance,0.751 ± 0.036,0.932 ± 0.017,0.733 ± 0.012,0.925 ± 0.001,0.817 ± 0.013,0.824 ± 0.056
flight,0.614 ± 0.019,0.907 ± 0.026,0.826 ± 0.035,0.907 ± 0.013,0.924 ± 0.021,0.878 ± 0.014
macro,0.693 ± 0.016,0.917 ± 0.020,0.768 ± 0.051,0.918 ± 0.008,0.835 ± 0.005,0.858 ± 0.007
